In [ ]:
import pandas as pd
import csv

# 定义文件配置列表
# 每个配置是一个字典，包含：
#   - path: 文件路径
#   - sep: 分隔符（如 ',' 或 '\t'），默认为 ','
#   - tweet_column: 推文所在的列名，默认为 'Tweet'（如果文件没有表头，需额外处理）
#   - header: 是否有表头，默认为 0（第一行是列名），若无表头则设为 None
#   - encoding_try: 可自定义尝试的编码顺序（可选）
file_configs = [
    {   
        ## 印尼印尼语辱骂性语言的数据集id-abusive-language-detection,2016条
        "path": "re_dataset_three_labels.csv",
        "sep": ",",
        "tweet_column": "Tweet",
        "header": 0
    },
    # {   # 印尼语仇恨言论检测数据集,该数据集包含 713 条印尼语推文
    #     "path": "/home/ninini/Dataset-LLM/Datasets/Indonesia/id-hatespeech-detection/IDHSD_RIO_unbalanced_713_2017.txt",
    #     "sep": "\t",
    #     "tweet_column": "Tweet",
    #     "header": 0
    # },
    # {
    #     #13169条
    #     "path": "/home/ninini/Dataset-LLM/Datasets/Indonesia/id-multi-label-hate-speech-and-abusive-language-detection/re_dataset.csv",
    #     "sep": ",",
    #     "tweet_column": "Tweet",
    #     "header": 0
    # },
    # {
    #     #14306条
    #     "path": "/home/ninini/Dataset-LLM/Datasets/Indonesia/indonesian-hate-speech-superset/in_hf.csv",
    #     "sep": ",",
    #     "tweet_column": "text",
    #     "header": 0
    # },
    # # 您可以继续添加更多文件，例如：
    # {
    #     # 7597条
    #     "path": "/home/ninini/Dataset-LLM/Datasets/Thailand/HateThaiSent/HateThaiSent.csv",
    #     "sep": ",",
    #     "tweet_column": "Message",
    #     "header": 0
    # },
    # {
    #     # 45000
    #     "path": "/home/ninini/Dataset-LLM/HateDay/split_by_country/en.csv",
    #     "sep": ",",
    #     "tweet_column": "text",
    #     "header": 0
    # },
    # {
    #     # 45000
    #     "path": "/home/ninini/Dataset-LLM/HateDay/split_by_country/in.csv",
    #     "sep": ",",
    #     "tweet_column": "text",
    #     "header": 0
    # },
    # {
    #     # 45000
    #     "path": "/home/ninini/Dataset-LLM/HateDay/split_by_country/US.csv",
    #     "sep": ",",
    #     "tweet_column": "text",
    #     "header": 0
    # },
    # {
    #     # 132
    #     "path": "/home/ninini/Dataset-LLM/Datasets/Singapore/Rabakbench/rabakbench_en.csv",
    #     "sep": ",",
    #     "tweet_column": "text",
    #     "header": 0
    # },
    # {
    #     # 132
    #     "path": "/home/ninini/Dataset-LLM/Datasets/Singapore/Rabakbench/rabakbench_ms.csv",
    #     "sep": ",",
    #     "tweet_column": "text",
    #     "header": 0
    # },
    # {   # 132
    #     "path": "/home/ninini/Dataset-LLM/Datasets/Singapore/Rabakbench/rabakbench_ta.csv",
    #     "sep": ",",
    #     "tweet_column": "text",
    #     "header": 0
    # },
    # {
    #     # 132
    #     "path": "/home/ninini/Dataset-LLM/Datasets/Singapore/Rabakbench/rabakbench_zh.csv",
    #     "sep": ",",
    #     "tweet_column": "text",
    #     "header": 0
    # }
]

def read_file_with_fallback(file_config):
    """
    根据配置尝试多种编码读取文件，返回DataFrame
    """
    file_path = file_config["path"]
    sep = file_config.get("sep", ",")
    header = file_config.get("header", 0)  # 默认第一行为列名
    encoding_list = file_config.get("encoding_try", ['utf-8', 'utf-8-sig', 'latin-1', 'cp1252'])
    
    for enc in encoding_list:
        try:
            df = pd.read_csv(file_path, sep=sep, header=header, encoding=enc)
            print(f"  ✓ 使用编码 {enc} 成功读取 {file_path}")
            return df
        except UnicodeDecodeError:
            continue
        except Exception as e:
            print(f"  ✗ 使用编码 {enc} 读取失败: {e}")
            continue
    # 如果所有编码都失败，使用 latin-1 强制读取（忽略错误）
    print(f"  ⚠ 所有编码均失败，使用 latin-1 强制读取（可能含乱码）")
    return pd.read_csv(file_path, sep=sep, header=header, encoding='latin-1')

def extract_tweets(df, tweet_column):
    """从DataFrame中提取推文列，去空，返回列表"""
    if tweet_column in df.columns:
        tweets = df[tweet_column].dropna().tolist()
        print(f"  提取到 {len(tweets)} 条推文")
        return tweets
    else:
        print(f"  ✗ 未找到列 '{tweet_column}'，实际列名: {df.columns.tolist()}")
        return []

def main():
    all_tweets = []
    for idx, config in enumerate(file_configs, 1):
        print(f"\n处理文件 {idx}: {config['path']}")
        df = read_file_with_fallback(config)
        tweets = extract_tweets(df, config["tweet_column"])
        all_tweets.extend(tweets)
    
    # 汇总保存
    output_df = pd.DataFrame({'tweet': all_tweets})
    output_file = "all_datasets_summary.csv"
    output_df.to_csv(output_file, index=False, encoding='utf-8', quoting=csv.QUOTE_MINIMAL)
    print(f"\n✓ 总计提取 {len(all_tweets)} 条推文，已保存至 {output_file}")
    print("CSV 已采用标准转义方式：包含逗号或引号的文本将被自动用双引号括起。")
    print("\n前5条预览：")
    print(output_df.head())

if __name__ == "__main__":
    main()

In [ ]:
import pandas as pd
import csv

# ========== 全局去重配置 ==========
DEDUP_KEEP = 'first'   # 'first' 或 'last'

# ========== 标签合并函数 ==========
def combine_labels_by_any(df, label_columns):
    """多列二值标签（0/1）：只要任意一列为1，结果即为1，否则0"""
    existing = [col for col in label_columns if col in df.columns]
    if not existing:
        return pd.Series([''] * len(df))
    combined = df[existing].max(axis=1).fillna(0).astype(int)
    return combined

# ========== 文件配置列表 ==========
file_configs = [
    {
        "path": "Indonesia.csv",
        "sep": ",",
        "tweet_column": "text",
        "label_config": {
            "columns": [
                "Hate_Speech",
                "Targeted_Harassment",
                "NSFW_Sexual",
                "Violence_Incitement",
                "Dangerous_Ideology",
                "Profanity_Slang"
            ],
            "func": combine_labels_by_any   # 使用默认合并函数
        },
        "header": 0
    },
    # 其他文件示例（单列标签）
    {
        "path": "re_dataset_three_labels.csv",
        "sep": ",",
        "tweet_column": "Tweet",
        "label_config": "Label",   # 直接写列名
        "header": 0
    },
    {
        "path": "in_hf.csv",
        "sep": ",",
        "tweet_column": "text",
        "label_config": "labels",   # 直接写列名
        "header": 0
    },
    {
        "path": "IDHSD_RIO_unbalanced_713_2017.csv",
        "sep": ",",
        "tweet_column": "Tweet",
        "label_config": "Label",   # 直接写列名
        "header": 0
    },
    {
        "path": "id.csv",
        "sep": ",",
        "tweet_column": "text",
        "label_config": "label",   # 直接写列名
        "header": 0
    },
    {
        "path": "re_dataset.csv",
        "sep": ",",
        "tweet_column": "Tweet",
        "label_config": {
            "columns": [
                "HS",
                "Abusive",
                "HS_Individual",
                "HS_Group",
                "HS_Religion",
                "HS_Race",
                "HS_Physical",
                "HS_Gender",
                "HS_Other",
                "HS_Weak",
                "HS_Moderate",
                "HS_Strong"

            ],
            "func": combine_labels_by_any   # 使用默认合并函数
        },
        "header": 0
    },
    {
        "path": "processed_train.csv",
        "sep": ",",
        "tweet_column": "processed_text",
        "label_config": {
            "columns": [
                "pornografi",
                "sara",
                "radikalisme",
                "pencemaran_nama_baik"
            ],
            "func": combine_labels_by_any   # 使用默认合并函数
        },
        "header": 0
    },
    {
        "path": "processed_test.csv",
        "sep": ",",
        "tweet_column": "processed_text",
        "label_config": {
            "columns": [
                "pornografi",
                "sara",
                "radikalisme",
                "pencemaran_nama_baik"
            ],
            "func": combine_labels_by_any   # 使用默认合并函数
        },
        "header": 0
    },
]

# ========== 读取与提取函数 ==========
def read_file_with_fallback(file_config):
    file_path = file_config["path"]
    sep = file_config.get("sep", ",")
    header = file_config.get("header", 0)
    encoding_list = file_config.get("encoding_try", ['utf-8', 'utf-8-sig', 'latin-1', 'cp1252'])
    for enc in encoding_list:
        try:
            df = pd.read_csv(file_path, sep=sep, header=header, encoding=enc)
            print(f"  ✓ 使用编码 {enc} 成功读取 {file_path}")
            return df
        except UnicodeDecodeError:
            continue
        except Exception as e:
            print(f"  ✗ 使用编码 {enc} 读取失败: {e}")
            continue
    print(f"  ⚠ 所有编码均失败，使用 latin-1 强制读取")
    return pd.read_csv(file_path, sep=sep, header=header, encoding='latin-1')

def extract_tweets_and_labels(df, tweet_column, label_config):
    """提取推文和标签，支持单列字符串或多列字典配置"""
    if tweet_column not in df.columns:
        print(f"  ✗ 未找到推文列 '{tweet_column}'，实际列名: {df.columns.tolist()}")
        return [], []
    
    # 删除推文为空的记录
    non_empty = df[tweet_column].notna()
    valid_df = df[non_empty].copy()
    tweets = valid_df[tweet_column].tolist()

    # 处理标签
    if label_config is None:
        labels = [''] * len(tweets)
    elif isinstance(label_config, str):
        # 旧式：单一列名
        if label_config not in df.columns:
            print(f"  ⚠ 未找到标签列 '{label_config}'，全部填充为空")
            labels = [''] * len(tweets)
        else:
            labels = valid_df[label_config].fillna('').astype(str).tolist()
    elif isinstance(label_config, dict):
        cols = label_config.get('columns', [])
        func = label_config.get('func', combine_labels_by_any)
        # 在原始 df 上计算标签（避免切片后索引失效）
        combined = func(df, cols)
        labels = combined[non_empty].fillna('').astype(str).tolist()
    else:
        raise TypeError(f"label_config 类型错误: {type(label_config)}")

    print(f"  提取到 {len(tweets)} 条推文，有效标签数: {len([l for l in labels if l != ''])}")
    return tweets, labels

def main():
    all_tweets = []
    all_labels = []

    for idx, config in enumerate(file_configs, 1):
        print(f"\n处理文件 {idx}: {config['path']}")
        df = read_file_with_fallback(config)
        tweets, labels = extract_tweets_and_labels(
            df,
            config["tweet_column"],
            config.get("label_config")  # 注意这里用的是 label_config
        )
        all_tweets.extend(tweets)
        all_labels.extend(labels)

    # 保存原始合并数据
    raw_df = pd.DataFrame({'tweet': all_tweets, 'label': all_labels})
    raw_output = "all_datasets_with_labels.csv"
    raw_df.to_csv(raw_output, index=False, encoding='utf-8', quoting=csv.QUOTE_MINIMAL)
    print(f"\n✓ 原始合并: {len(raw_df)} 条记录，已保存至 {raw_output}")

    # 去重（基于 tweet）
    dedup_df = raw_df.drop_duplicates(subset=['tweet'], keep=DEDUP_KEEP)
    removed = len(raw_df) - len(dedup_df)
    print(f"✓ 去重后: {len(dedup_df)} 条记录（移除 {removed} 条重复）")

    dedup_output = "all_datasets_with_labels_dedup.csv"
    dedup_df.to_csv(dedup_output, index=False, encoding='utf-8', quoting=csv.QUOTE_MINIMAL)
    print(f"✓ 去重结果已保存至 {dedup_output}")
    print("\n去重后前5条预览：")
    print(dedup_df.head())

if __name__ == "__main__":
    main()


处理文件 1: Indonesia.csv
  ✓ 使用编码 utf-8 成功读取 Indonesia.csv
  提取到 30201 条推文，有效标签数: 30201

处理文件 2: re_dataset_three_labels.csv
  ✓ 使用编码 utf-8 成功读取 re_dataset_three_labels.csv
  提取到 2016 条推文，有效标签数: 2016

处理文件 3: in_hf.csv
  ✓ 使用编码 utf-8 成功读取 in_hf.csv
  提取到 14306 条推文，有效标签数: 14306

处理文件 4: IDHSD_RIO_unbalanced_713_2017.csv
  ✓ 使用编码 utf-8 成功读取 IDHSD_RIO_unbalanced_713_2017.csv
  提取到 713 条推文，有效标签数: 713

处理文件 5: re_dataset.csv
  ✓ 使用编码 latin-1 成功读取 re_dataset.csv
  提取到 13169 条推文，有效标签数: 13169

处理文件 6: processed_train.csv
  ✓ 使用编码 utf-8 成功读取 processed_train.csv
  提取到 6995 条推文，有效标签数: 6995

处理文件 7: processed_test.csv
  ✓ 使用编码 utf-8 成功读取 processed_test.csv
  提取到 778 条推文，有效标签数: 778

✓ 原始合并: 68178 条记录，已保存至 all_datasets_with_labels.csv
✓ 去重后: 21505 条记录（移除 46673 条重复）
✓ 去重结果已保存至 all_datasets_with_labels_dedup.csv

去重后前5条预览：
                                               tweet label
0  - Dia sendiri yang ngiklanin promo cashback di...     1
1  - disaat semua cowok berusaha melacak perhatia...     1
2  - k